# Bienvenido/a al Entorno de Desarrollo
## Prueba Técnica — Analytics Engineer Senior

Este notebook verifica que todas las herramientas estén correctamente instaladas
y que los datasets estén disponibles. Ejecútalo completo antes de comenzar.

---

## 1. Verificación de dependencias

In [ ]:
import sys
import importlib

libs = {
    'pandas':     'pandas',
    'numpy':      'numpy',
    'pyarrow':    'pyarrow',
    'duckdb':     'duckdb',
    'pyspark':    'pyspark',
    'matplotlib': 'matplotlib',
    'seaborn':    'seaborn',
    'plotly':     'plotly',
}

print(f'Python {sys.version}\n')
all_ok = True
for name, module in libs.items():
    try:
        m = importlib.import_module(module)
        version = getattr(m, '__version__', 'n/a')
        print(f'  ✅  {name:<15} {version}')
    except ImportError:
        print(f'  ❌  {name:<15} NO ENCONTRADO')
        all_ok = False

print()
if all_ok:
    print('✅ Todas las dependencias están disponibles.')
else:
    print('❌ Algunas dependencias faltan. Revisa el README.')

## 2. Verificación de datasets

In [ ]:
import os
import pandas as pd

RAW_PATH = '/workspace/data/raw'

datasets = {
    'clientes_cdmx.csv':           'csv',
    'clientes_gdl_mty.csv':        'csv',
    'clientes_resto.parquet':      'parquet',
    'catalogo_productos.csv':      'csv',
    'ordenes_2022_2023.parquet':   'parquet',
    'ordenes_2024.parquet':        'parquet',
    'devoluciones.txt':            'txt',
}

print(f'Buscando archivos en: {RAW_PATH}\n')
all_found = True

for fname, ftype in datasets.items():
    fpath = os.path.join(RAW_PATH, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        try:
            if ftype == 'csv':
                df = pd.read_csv(fpath, nrows=5)
            elif ftype == 'parquet':
                df = pd.read_parquet(fpath).head(5)
            else:
                df = pd.read_csv(fpath, sep='|', nrows=5)
            print(f'  ✅  {fname:<42} {size_mb:>6.1f} MB   {len(df.columns)} columnas')
        except Exception as e:
            print(f'  ⚠️   {fname:<42} Error al leer: {e}')
    else:
        print(f'  ❌  {fname:<42} NO ENCONTRADO')
        all_found = False

print()
if all_found:
    print('✅ Todos los datasets están disponibles.')
else:
    print('❌ Algunos archivos faltan. Cópialos a data/raw/ y vuelve a ejecutar.')

## 3. Verificación de DuckDB

In [ ]:
import duckdb

WH_PATH = '/workspace/data/warehouse/mercado.duckdb'

try:
    con = duckdb.connect(WH_PATH)
    result = con.execute('SELECT 42 AS test').fetchone()
    con.close()
    print(f'✅ DuckDB operativo. Warehouse en: {WH_PATH}')
    print(f'   Prueba de conexión: SELECT 42 → {result[0]}')
except Exception as e:
    print(f'❌ Error con DuckDB: {e}')

## 4. Verificación de PySpark

In [ ]:
from pyspark.sql import SparkSession

try:
    spark = SparkSession.builder \
        .appName('MercadoAnalytics_Test') \
        .master('local[2]') \
        .config('spark.ui.showConsoleProgress', 'false') \
        .getOrCreate()

    spark.sparkContext.setLogLevel('ERROR')
    test_df = spark.createDataFrame([(1, 'ok')], ['id', 'status'])
    count = test_df.count()
    spark.stop()
    print(f'✅ PySpark operativo. Versión: {spark.version}')
    print(f'   Prueba de DataFrame: {count} fila(s) procesada(s)')
except Exception as e:
    print(f'❌ Error con PySpark: {e}')

## 5. Vista rápida de los datasets

Una muestra de las primeras filas de cada archivo para que te orientes antes de comenzar el EDA.

In [ ]:
import pandas as pd

RAW = '/workspace/data/raw'

print('── clientes_cdmx.csv ──')
display(pd.read_csv(f'{RAW}/clientes_cdmx.csv', nrows=3))

print('\n── clientes_gdl_mty.csv ──')
display(pd.read_csv(f'{RAW}/clientes_gdl_mty.csv', nrows=3))

print('\n── clientes_resto.parquet ──')
display(pd.read_parquet(f'{RAW}/clientes_resto.parquet').head(3))

print('\n── catalogo_productos.csv ──')
display(pd.read_csv(f'{RAW}/catalogo_productos.csv', nrows=3))

print('\n── ordenes_2022_2023.parquet (primeras 3 filas) ──')
display(pd.read_parquet(f'{RAW}/ordenes_2022_2023.parquet').head(3))

print('\n── ordenes_2024.parquet (primeras 3 filas) ──')
display(pd.read_parquet(f'{RAW}/ordenes_2024.parquet').head(3))

print('\n── devoluciones.txt (primeras 3 filas) ──')
display(pd.read_csv(f'{RAW}/devoluciones.txt', sep='|', nrows=3))

---

## ✅ Todo listo

Si todas las celdas anteriores mostraron `✅`, tu entorno está correctamente configurado.

**Estructura sugerida para tu trabajo:**

```
notebooks/
  my_notebook.ipynb

scripts/
  my_script.py

sql/
  example.sql

dbt_project/
  models/
    staging/
    intermediate/
    marts/
```

¡Mucho éxito!
